In [ ]:
import tilelang
import tilelang.language as T

@tilelang.jit
def elementwise_add(A, B, block_M, block_N, in_dtype, out_dtype, threads):
    M, N = T.const("M, N")

    A: T.Tensor((M, N), in_dtype) # type: ignore
    B: T.Tensor((M, N), in_dtype) # type: ignore
    C = T.empty((M, N), out_dtype)

    with T.Kernel(T.ceildiv(N, block_N), T.ceildiv(M, block_M), threads=threads) as (bx, by):
        A_shared = T.alloc_shared((block_M, block_N), in_dtype)
        B_shared = T.alloc_shared((block_M, block_N), in_dtype)
        C_local = T.alloc_fragment((block_M, block_N), out_dtype)
        C_shared = T.alloc_shared((block_M, block_N), out_dtype)

        T.copy(A[by * block_M, bx * block_N], A_shared)
        T.copy(B[by * block_M, bx * block_N], B_shared)
        for local_y, local_x in T.Parallel(block_M, block_N):
            C_local[local_y, local_x] = A_shared[local_y, local_x] + B_shared[local_y, local_x]
        T.copy(C_local, C_shared)
        T.copy(C_shared, C[by * block_M, bx * block_N])

    return C

import torch

def ref_program(x, y):
    return x + y

M, N = 1024, 1024
a = torch.randn(M, N, dtype=torch.float32, device="cuda")
b = torch.randn(M, N, dtype=torch.float32, device="cuda")

cuda_src = elementwise_add.get_kernel_source(
    a, b, block_M=32, block_N=32, threads=128,
    in_dtype="float32", out_dtype="float32",
)
# print(cuda_src)

kernel = elementwise_add.compile(a, b, block_M=32, block_N=32, threads=128, in_dtype="float32", out_dtype="float32")
print(kernel.get_kernel_source())    # kernel 源码
# print(kernel.get_host_source())     # host C/C++ 源码
# kernel.show_source("both")          # 打印两者
# kernel.export_sources("/tmp/kernel.cu", "/tmp/host.cc")
# kernel.show_ptx()
# kernel.show_sass()
# kernel.export_library("/tmp/kernel.so")


2026-07-07 00:40:30  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:357): Found kernel 'elementwise_add' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.
2026-07-07 00:40:30  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:357): Found kernel 'elementwise_add' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.
#if defined(_MSC_VER) && !defined(__clang__) && _MSC_VER < 1940
#define _tl_orig_alignas alignas
#define alignas(N) _tl_orig_alignas((N) <= 64 ? (N) : 64)
#include <cuda.h>
#undef alignas
#define alignas _tl_orig_alignas
#endif
#include <tl_templates/cuda/intrin.h>
#include <tl_templates/cuda/barrier.h>
#include <tl_templates/cuda/copy_sm90.h>
#include <tl_templates/cuda/reduce.h>
#include <tl_templates/cuda/scan.h>
#include <tl_templates/cuda/ldsm.h>
#include <tl_templates/cuda/threadblock_swizzle.h>
#include <tl_templates/cuda/debug.h>


In [ ]:
import torch

def ref_program(x, y):
    return x + y

M, N = 1024, 1024
a = torch.randn(M, N, dtype=torch.float32, device="cuda")
b = torch.randn(M, N, dtype=torch.float32, device="cuda")

out = elementwise_add(a, b, block_M=32, block_N=32, threads=128, in_dtype=T.float32, out_dtype=T.float32)

torch.testing.assert_close(out, ref_program(a, b), rtol=1e-2, atol=1e-2)